In [1]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import sys
from typing import Dict, Any, List, Optional, Iterable

import numpy as np
import ollama
from sentence_transformers import SentenceTransformer
from opensearchpy import OpenSearch, helpers


%load_ext autoreload
%autoreload 2

from src.constants import(
    ASSYMETRIC_EMBEDDING,
    OLLAMA_MODEL_NAME,
    OPENSEARCH_HOST,
    OPENSEARCH_PORT,
    OPENSEARCH_INDEX
)

c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIMENSION = 384
ASSYMETRIC_EMBEDDING = False

In [4]:
client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_compress=True,
    timeout=30,
    max_retries=3,
    retry_on_timeout=True,
)
try:
    info = client.info()
    print(f"Successfully connected to OpenSearch {info['version']['number']}")
except Exception as e:
    print(f"Failed to connect to OpenSearch: {e}")
    print("Make sure OpenSearch is running on localhost:9200")
    raise

Successfully connected to OpenSearch 2.19.2


In [5]:
from opensearchpy.exceptions import NotFoundError
pipeline_name = "nlp-search-pipeline"

try:
    result = client.transport.perform_request(
        "GET",
        f"/_search/pipeline/{pipeline_name}"
    )
    print(f"\n Search pipeline {pipeline_name} exists.")
    print(result)
except NotFoundError:
    print(f"\n Search pipeline {pipeline_name} not exists.")
except Exception as e:
    print(f"\n error {e}")


 Search pipeline nlp-search-pipeline exists.
{'nlp-search-pipeline': {'description': 'Post processor for hybrid search', 'phase_results_processors': [{'normalization-processor': {'normalization': {'technique': 'min_max'}, 'combination': {'technique': 'arithmetic_mean', 'parameters': {'weights': [0.3, 0.7]}}}}]}}


In [6]:
def hybrid_search(query_text: str, query_embedding: List[float], top_k: int = 5) -> List[Dict[str, Any]]:
    """
    Performs hybrid search combining text-based and vector-based queries.
    
    Args:
        query_text (str): The text query for BM25 search
        query_embedding (List[float]): The vector embedding for KNN search
        top_k (int): Number of results to return
        
    Returns:
        List[Dict[str, Any]]: The search results
    """
    query_body = {
        "_source": {"exclude": ["embedding"]},
        "query": {
            "hybrid": {
                "queries": [
                    {"match": {"text": {"query": query_text}}},
                    {
                        "knn": {
                            "embedding": {
                                "vector": query_embedding,
                                "k": top_k,
                            }
                        }
                    },
                ]
            }
        },
        "size": top_k,
    }
    try:
        response = client.search(
            index=OPENSEARCH_INDEX,
            body=query_body,
            params={"search_pipeline": "nlp-search-pipeline"}
        )
    except TypeError:
        print("Warning: OpenSearch client doesn't support search_pipeline parameter, using raw query")
        response = client.search(
            index=OPENSEARCH_INDEX,
            body=query_body
        )
    
    return response["hits"]["hits"]

In [7]:
def get_embedding_model():
    """
    Loads and returns the sentence transformer embedding model.
    
    Returns:
        SentenceTransformer: The loaded embedding model.
    """
    print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    return model


def generate_embeddings(texts: List[str]):
    """
    Generates embeddings for a list of text chunks.
    
    Args:
        texts (List[str]): List of text chunks to embed.
        
    Returns:
        List[numpy.ndarray]: List of embedding vectors.
    """
    model = get_embedding_model()

    if ASSYMETRIC_EMBEDDING:
        texts = [f"passage: {text}" for text in texts]
        
    embeddings = model.encode(texts)
    return embeddings

In [8]:
query = "What is attention?"
print(f"Query: '{query}'")

print("\nGenerating embedding for query...")
embeddings = generate_embeddings([query])
query_embedding = embeddings[0].tolist()
print(f"Generated embedding with dimension: {len(query_embedding)}")

top_k = 3
print(f"\nRetrieving top {top_k} documents...")

results = hybrid_search(query, query_embedding, top_k=top_k)

print(f"\nSearch results for query: '{query}'\n")
for i, hit in enumerate(results, 1):
    print(f"Result {i} (Score: {hit['_score']:.3f}):")
    print(f"Text: {hit['_source']['text'][:200]}...")
    print(f"Document: {hit['_source']['document_name']}\n")


Query: 'What is attention?'

Generating embedding for query...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2461.67it/s]


Generated embedding with dimension: 384

Retrieving top 3 documents...

Search results for query: 'What is attention?'

Result 1 (Score: 0.700):
Text: of contiguous kernels, orO(logk(n))in the case of dilated convolutions [ 18], increasing the length of the longest paths between any two positions in the network. Convolutional layers are generally mo...
Document: attention is all you need

Result 2 (Score: 0.301):
Text: An example of the attention mechanism following long-distance dependencies in the encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of the verb ‘making...
Document: attention is all you need

Result 3 (Score: 0.204):
Text: , pages 433–440. ACL, July 2006. [30] Ofir Press and Lior Wolf. Using the output embedding to improve language models. arXiv preprint arXiv:1608.05859 , 2016. [31] Rico Sennrich, Barry Haddow, and Ale...
Document: attention is all you need



In [9]:
def generate_response_with_ollama(query: str, results: List[Dict], model_name: str = OLLAMA_MODEL_NAME):
    """
    Generates a response using Ollama based on search results.
    
    Args:
        query (str): The user's question
        results (List[Dict]): The search results from OpenSearch
        model_name (str): The Ollama model to use
        
    Returns:
        tuple: A tuple containing (prompt, model_name)
    """
    context = ""
    for i, result in enumerate(results):
        context += f"Document {i + 1}:\n{result['_source']['text']}\n\n"

    prompt = f"""You are a helpful AI assistant. Use the following context to answer the question.
If you cannot find the answer in the context, say so.

Context:
{context}

Question: {query}

Answer: """

    return prompt, model_name

In [10]:
print(f"Ensuring Ollama model {OLLAMA_MODEL_NAME} is available...")
try:
    ollama.pull(OLLAMA_MODEL_NAME)
    print(f"Model {OLLAMA_MODEL_NAME} is ready.")
except ollama.ResponseError as e:
    print(f"Error pulling model: {e.error}")
    print("You might need to install the model manually with: ollama pull " + OLLAMA_MODEL_NAME)

prompt, model_name = generate_response_with_ollama(query, results)

Ensuring Ollama model qwen3:4b is available...
Model qwen3:4b is ready.


In [11]:
print("\nGenerating response with Ollama...")
response = ""
print("\nResponse:")
for chunk in ollama.generate(model=model_name, prompt=prompt, stream=True):
    piece = chunk['response']
    print(piece, end='', flush=True)
    response += piece

print("\n\nResponse generation complete!")
print(f"Generated {len(response)} characters")


Generating response with Ollama...

Response:
Based on the provided context, attention refers to the self-attention mechanism in transformer models that allows the model to focus on relevant parts of the input sequence. 

Specifically, the context describes that "many of the attention heads appear to exhibit behavior related to the syntactic and semantic structure of the sentences." The attention mechanism helps the model capture long-distance dependencies in text, as evidenced by examples like "Many of the attention heads attend to a distant dependency of the verb 'making', completing the phrase 'making...more difficult'."

In the transformer architecture, attention is implemented through multiple "attention heads" (subcomponents) that each focus on different aspects of the input. The context mentions that "individual attention heads clearly learn to perform different tasks," indicating that attention is a key component that enables the model to understand both syntactic and semantic